# Relatório de Estrutura da Base

Notebook para gerar o relatório HTML do ETL + estrutura das variáveis, com botões para baixar tabelas em CSV e exportação automática de tabelas `.tex` para LaTeX.

In [2]:
# ============================================================
# RELATÓRIO HTML DO ETL + ESTRUTURA DAS VARIÁVEIS
# COM BOTÕES DE DOWNLOAD + EXPORTAÇÃO PARA LaTeX
#
# Entrada:
# - a.csv
#
# Saídas:
# - creditcard.csv
# - relatorio_estrutura_base.html
# - relatorio_estrutura_base/
#     ├── tabela_distribuicao_target.tex
#     ├── tabela_resumo_tcc.tex
#     ├── tabela_estrutura_variaveis.tex
#     ├── tabela_primeiras_linhas.tex
#     ├── tabela_resumo_etl.tex
#     ├── tabela_transformacoes.tex
#     └── comandos_latex_exemplo.tex
# ============================================================

import html
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# FUNÇÕES AUXILIARES DE FORMATAÇÃO
# ============================================================

def formatar_int(valor):
    """
    Formata inteiro com separador de milhar no padrão brasileiro.
    """
    try:
        return f"{int(valor):,}".replace(",", ".")
    except Exception:
        return str(valor)


def formatar_float(valor, casas=4):
    """
    Formata float com vírgula decimal para exibição no HTML.
    """
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_percentual(valor, casas=4):
    """
    Formata percentual para HTML.
    """
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}%".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=4):
    """
    Formata float para LaTeX mantendo ponto decimal.
    """
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def limpar_nome_arquivo(nome):
    """
    Limpa string para ser usada como nome de arquivo.
    """
    nome = str(nome)
    nome = nome.replace(" ", "_")
    nome = "".join(c for c in nome if c.isalnum() or c in ["_", "-", "."])
    return nome


# ============================================================
# FUNÇÕES AUXILIARES PARA HTML
# ============================================================

def gerar_card(titulo, valor, descricao=None):
    """
    Gera card visual para métricas do HTML.
    """
    desc_html = ""

    if descricao is not None:
        desc_html = f'<div class="card-desc">{descricao}</div>'

    return f"""
    <div class="metric-card">
        <div class="card-title">{html.escape(str(titulo))}</div>
        <div class="card-value">{html.escape(str(valor))}</div>
        {desc_html}
    </div>
    """


def gerar_tabela_html(df, table_id, classe="data-table", max_linhas=None):
    """
    Converte DataFrame para tabela HTML estilizada com ID,
    permitindo baixar a tabela pelo botão do HTML.
    """
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    df_html = df.copy()

    if max_linhas is not None:
        df_html = df_html.head(max_linhas).copy()

    html_tabela = f'<table id="{html.escape(str(table_id))}" class="{classe}">\n'

    html_tabela += "<thead><tr>"
    for col in df_html.columns:
        html_tabela += f"<th>{html.escape(str(col))}</th>"
    html_tabela += "</tr></thead>\n"

    html_tabela += "<tbody>\n"

    for _, row in df_html.iterrows():
        html_tabela += "<tr>"
        for valor in row:
            html_tabela += f"<td>{html.escape(str(valor))}</td>"
        html_tabela += "</tr>\n"

    html_tabela += "</tbody></table>"

    return html_tabela


def gerar_secao_tabela(titulo, tabela_html, table_id, nome_csv):
    """
    Gera seção HTML com título, botão de download CSV e tabela.
    """
    return f"""
    <section class="section">
        <div class="section-header">
            <h2>{html.escape(str(titulo))}</h2>
            <button
                class="download-btn"
                onclick="baixarTabelaCSV('{html.escape(str(table_id))}', '{html.escape(str(nome_csv))}')"
            >
                Baixar CSV
            </button>
        </div>

        <div class="table-wrapper">
            {tabela_html}
        </div>
    </section>
    """


# ============================================================
# FUNÇÕES AUXILIARES PARA LaTeX
# ============================================================

def preparar_df_latex(df):
    """
    Prepara DataFrame para exportação em LaTeX.
    Mantém ponto decimal, pois é mais seguro em LaTeX.
    """
    if df is None or df.empty:
        return pd.DataFrame()

    df_latex = df.copy()

    for col in df_latex.columns:
        if pd.api.types.is_float_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: formatar_float_latex(x, 4))
        elif pd.api.types.is_integer_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_latex


def salvar_tabela_latex(
    df,
    caminho,
    caption,
    label,
    longtable=True
):
    """
    Salva DataFrame em formato .tex.

    Pacotes recomendados no LaTeX:
        \\usepackage{booktabs}
        \\usepackage{longtable}
        \\usepackage{float}
        \\usepackage{graphicx}
        \\usepackage{pdflscape}
    """
    caminho = Path(caminho)

    if df is None or df.empty:
        caminho.write_text(
            "% Tabela vazia: nenhum dado disponível.\n",
            encoding="utf-8"
        )
        return

    df_latex = preparar_df_latex(df)

    tex = df_latex.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    caminho.write_text(tex, encoding="utf-8")


def exportar_artefatos_latex(
    pasta_latex,
    resumo_etl,
    tabela_transformacoes,
    resumo_status,
    resumo_tcc,
    estrutura_variaveis,
    primeiras_linhas
):
    """
    Exporta tabelas .tex e comandos de exemplo para LaTeX.
    """
    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    salvar_tabela_latex(
        resumo_etl,
        pasta_latex / "tabela_resumo_etl.tex",
        caption="Resumo geral do processo de tratamento da base.",
        label="tab:resumo-etl",
        longtable=False
    )

    salvar_tabela_latex(
        tabela_transformacoes,
        pasta_latex / "tabela_transformacoes.tex",
        caption="Transformações aplicadas durante o tratamento da base.",
        label="tab:transformacoes-base",
        longtable=False
    )

    salvar_tabela_latex(
        resumo_status,
        pasta_latex / "tabela_distribuicao_target.tex",
        caption="Distribuição da variável alvo após o tratamento da base.",
        label="tab:distribuicao-target",
        longtable=False
    )

    salvar_tabela_latex(
        resumo_tcc,
        pasta_latex / "tabela_resumo_tcc.tex",
        caption="Resumo agrupado da estrutura das variáveis para apresentação no TCC.",
        label="tab:resumo-tcc-estrutura",
        longtable=False
    )

    salvar_tabela_latex(
        estrutura_variaveis,
        pasta_latex / "tabela_estrutura_variaveis.tex",
        caption="Estrutura completa das variáveis da base tratada.",
        label="tab:estrutura-variaveis",
        longtable=True
    )

    salvar_tabela_latex(
        primeiras_linhas,
        pasta_latex / "tabela_primeiras_linhas.tex",
        caption="Primeiras linhas da base tratada.",
        label="tab:primeiras-linhas-base",
        longtable=True
    )

    comandos_latex = r"""
% ============================================================
% COMANDOS LaTeX DE EXEMPLO
%
% Pacotes recomendados no preâmbulo:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}
% ============================================================

% ----------------------------
% Resumo geral do ETL
% ----------------------------
\begin{table}[H]
\centering
\caption{Resumo geral do tratamento da base.}
\label{tab:resumo-etl-main}
\input{relatorio_estrutura_base/tabela_resumo_etl.tex}
\end{table}

% ----------------------------
% Transformações aplicadas
% ----------------------------
\begin{table}[H]
\centering
\caption{Transformações aplicadas na base original.}
\label{tab:transformacoes-base-main}
\input{relatorio_estrutura_base/tabela_transformacoes.tex}
\end{table}

% ----------------------------
% Distribuição do target
% ----------------------------
\begin{table}[H]
\centering
\caption{Distribuição da variável alvo após tratamento.}
\label{tab:distribuicao-target-main}
\input{relatorio_estrutura_base/tabela_distribuicao_target.tex}
\end{table}

% ----------------------------
% Resumo agrupado para o TCC
% ----------------------------
\begin{table}[H]
\centering
\caption{Resumo agrupado das variáveis utilizadas.}
\label{tab:resumo-tcc-estrutura-main}
\input{relatorio_estrutura_base/tabela_resumo_tcc.tex}
\end{table}

% ----------------------------
% Estrutura completa das variáveis
% ----------------------------
% Como é uma tabela mais larga, recomenda-se usar landscape:
\begin{landscape}
\small
\input{relatorio_estrutura_base/tabela_estrutura_variaveis.tex}
\end{landscape}

% ----------------------------
% Primeiras linhas da base tratada
% ----------------------------
\begin{landscape}
\small
\input{relatorio_estrutura_base/tabela_primeiras_linhas.tex}
\end{landscape}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(
        comandos_latex,
        encoding="utf-8"
    )


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_relatorio_estrutura_base_html(
    arquivo_entrada="a.csv",
    arquivo_saida_base="creditcard.csv",
    arquivo_html_saida="relatorio_estrutura_base.html",
    pasta_saida=".",
    pasta_latex="relatorio_estrutura_base",
    exportar_latex=True
):
    """
    Gera relatório HTML completo do ETL inicial e da estrutura da base.

    Etapas:
    1. Lê a base original.
    2. Renomeia Time, Amount e Class.
    3. Remove duplicatas.
    4. Remove faltantes.
    5. Salva creditcard.csv.
    6. Cria relatório HTML com resumo do ETL e estrutura das variáveis.
    7. Exporta tabelas em .tex para LaTeX, se exportar_latex=True.
    """

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / arquivo_html_saida
    caminho_base_tratada = pasta_saida / arquivo_saida_base
    caminho_latex = pasta_saida / pasta_latex

    # ========================================================
    # 1. CARREGAR ARQUIVO ORIGINAL
    # ========================================================

    df_original = pd.read_csv(arquivo_entrada)

    linhas_iniciais = df_original.shape[0]
    colunas_iniciais = df_original.shape[1]

    colunas_originais = df_original.columns.tolist()

    # ========================================================
    # 2. RENOMEAR COLUNAS
    # ========================================================

    df = df_original.rename(
        columns={
            "Time": "tempo_desde_a_primeira_transacao",
            "Amount": "valor_de_transacao",
            "Class": "status_fraude"
        }
    )

    colunas_apos_renomeacao = df.columns.tolist()

    # ========================================================
    # 3. REMOVER DUPLICATAS
    # ========================================================

    linhas_antes_duplicatas = df.shape[0]
    duplicatas_identificadas = int(df.duplicated().sum())

    df = df.drop_duplicates(
        keep="first"
    ).reset_index(drop=True)

    linhas_depois_duplicatas = df.shape[0]
    linhas_removidas_duplicatas = (
        linhas_antes_duplicatas
        - linhas_depois_duplicatas
    )

    # ========================================================
    # 4. REMOVER DADOS FALTANTES
    # ========================================================

    linhas_antes_faltantes = df.shape[0]
    total_valores_faltantes = int(df.isnull().sum().sum())
    linhas_com_faltantes = int(df.isnull().any(axis=1).sum())

    df = df.dropna().reset_index(drop=True)

    linhas_depois_faltantes = df.shape[0]
    linhas_removidas_faltantes = (
        linhas_antes_faltantes
        - linhas_depois_faltantes
    )

    # ========================================================
    # 5. SALVAR BASE TRATADA
    # ========================================================

    df.to_csv(
        caminho_base_tratada,
        index=False,
        encoding="utf-8-sig"
    )

    linhas_finais = df.shape[0]
    colunas_finais = df.shape[1]
    linhas_removidas_total = linhas_iniciais - linhas_finais
    valores_faltantes_restantes = int(df.isnull().sum().sum())

    percentual_linhas_removidas = (
        100 * linhas_removidas_total / linhas_iniciais
        if linhas_iniciais > 0
        else 0
    )

    percentual_duplicatas_removidas = (
        100 * linhas_removidas_duplicatas / linhas_iniciais
        if linhas_iniciais > 0
        else 0
    )

    percentual_faltantes_removidas = (
        100 * linhas_removidas_faltantes / linhas_iniciais
        if linhas_iniciais > 0
        else 0
    )

    percentual_base_preservada = (
        100 * linhas_finais / linhas_iniciais
        if linhas_iniciais > 0
        else 0
    )

    # ========================================================
    # 6. RESUMO GERAL DO ETL COMO TABELA
    # ========================================================

    resumo_etl = pd.DataFrame([
        {
            "Métrica": "Linhas iniciais",
            "Valor": linhas_iniciais,
            "Descrição": "Quantidade de registros antes do tratamento."
        },
        {
            "Métrica": "Colunas iniciais",
            "Valor": colunas_iniciais,
            "Descrição": "Quantidade de variáveis no arquivo original."
        },
        {
            "Métrica": "Linhas finais",
            "Valor": linhas_finais,
            "Descrição": "Quantidade de registros após limpeza."
        },
        {
            "Métrica": "Colunas finais",
            "Valor": colunas_finais,
            "Descrição": "Quantidade de variáveis após renomeação."
        },
        {
            "Métrica": "Duplicatas identificadas",
            "Valor": duplicatas_identificadas,
            "Descrição": "Número de linhas duplicadas detectadas."
        },
        {
            "Métrica": "Removidas por duplicidade",
            "Valor": linhas_removidas_duplicatas,
            "Descrição": f"{percentual_duplicatas_removidas:.4f}% das linhas iniciais."
        },
        {
            "Métrica": "Linhas com faltantes",
            "Valor": linhas_com_faltantes,
            "Descrição": "Linhas com pelo menos um valor ausente."
        },
        {
            "Métrica": "Removidas por faltantes",
            "Valor": linhas_removidas_faltantes,
            "Descrição": f"{percentual_faltantes_removidas:.4f}% das linhas iniciais."
        },
        {
            "Métrica": "Total removido",
            "Valor": linhas_removidas_total,
            "Descrição": f"{percentual_linhas_removidas:.4f}% das linhas iniciais."
        },
        {
            "Métrica": "Base preservada",
            "Valor": f"{percentual_base_preservada:.4f}%",
            "Descrição": "Percentual da base original mantida após limpeza."
        },
        {
            "Métrica": "Valores faltantes restantes",
            "Valor": valores_faltantes_restantes,
            "Descrição": "Deve ser zero após o tratamento."
        },
        {
            "Métrica": "Dimensão final",
            "Valor": f"{linhas_finais} x {colunas_finais}",
            "Descrição": "Linhas x colunas."
        },
    ])

    tabela_transformacoes = pd.DataFrame([
        {
            "Etapa": 1,
            "Transformação": "Leitura da base original",
            "Descrição": f"Arquivo lido: {arquivo_entrada}"
        },
        {
            "Etapa": 2,
            "Transformação": "Renomeação de colunas",
            "Descrição": "Time, Amount e Class foram renomeadas para nomes descritivos."
        },
        {
            "Etapa": 3,
            "Transformação": "Remoção de duplicatas",
            "Descrição": "Linhas duplicadas foram removidas mantendo a primeira ocorrência."
        },
        {
            "Etapa": 4,
            "Transformação": "Remoção de valores faltantes",
            "Descrição": "Linhas com valores ausentes foram removidas."
        },
        {
            "Etapa": 5,
            "Transformação": "Exportação da base tratada",
            "Descrição": f"Base final salva como {arquivo_saida_base}."
        },
    ])

    # ========================================================
    # 7. DISTRIBUIÇÃO DO TARGET
    # ========================================================

    if "status_fraude" in df.columns:
        resumo_status_raw = (
            df["status_fraude"]
            .value_counts()
            .sort_index()
            .to_frame("quantidade")
        )

        resumo_status_raw["percentual"] = (
            100
            * resumo_status_raw["quantidade"]
            / resumo_status_raw["quantidade"].sum()
        )

        resumo_status_raw = resumo_status_raw.reset_index()
        resumo_status_raw.columns = ["classe", "quantidade", "percentual"]

        resumo_status_raw["classe"] = resumo_status_raw["classe"].map(
            {
                0: "Não Fraude",
                1: "Fraude"
            }
        ).fillna(resumo_status_raw["classe"].astype(str))

        resumo_status = resumo_status_raw.copy()
        resumo_status["quantidade"] = resumo_status["quantidade"].apply(formatar_int)
        resumo_status["percentual"] = resumo_status["percentual"].apply(
            lambda x: f"{x:.4f}%"
        )

    else:
        resumo_status_raw = pd.DataFrame(
            columns=["classe", "quantidade", "percentual"]
        )

        resumo_status = pd.DataFrame(
            columns=["classe", "quantidade", "percentual"]
        )

    # ========================================================
    # 8. ESTRUTURA DAS VARIÁVEIS
    # ========================================================

    estrutura_variaveis_raw = pd.DataFrame(
        {
            "Variável": df.columns,
            "Tipo Python": df.dtypes.astype(str).values,
            "Valores não nulos": df.notnull().sum().values,
            "Valores nulos": df.isnull().sum().values,
            "% nulos": (df.isnull().mean() * 100).round(4).values,
            "Valores únicos": df.nunique(dropna=True).values,
            "Memória bytes": df.memory_usage(
                deep=True,
                index=False
            ).values
        }
    )

    estrutura_variaveis_raw["Memória KB"] = (
        estrutura_variaveis_raw["Memória bytes"] / 1024
    ).round(2)

    estrutura_variaveis_raw["Memória MB"] = (
        estrutura_variaveis_raw["Memória bytes"] / (1024 ** 2)
    ).round(4)

    estrutura_variaveis_formatada = estrutura_variaveis_raw.copy()

    for col in ["Valores não nulos", "Valores nulos", "Valores únicos", "Memória bytes"]:
        estrutura_variaveis_formatada[col] = estrutura_variaveis_formatada[col].apply(formatar_int)

    estrutura_variaveis_formatada["% nulos"] = estrutura_variaveis_formatada["% nulos"].apply(
        lambda x: f"{x:.4f}%"
    )

    estrutura_variaveis_formatada["Memória KB"] = estrutura_variaveis_formatada["Memória KB"].apply(
        lambda x: f"{x:.2f}"
    )

    estrutura_variaveis_formatada["Memória MB"] = estrutura_variaveis_formatada["Memória MB"].apply(
        lambda x: f"{x:.4f}"
    )

    # ========================================================
    # 9. RESUMO AGRUPADO PARA TCC
    # ========================================================

    linhas_resumo_tcc = []

    if "V1" in df.columns:
        linhas_resumo_tcc.append(
            {
                "Variável": "V1 a V28",
                "Tipo": str(df["V1"].dtype),
                "Não nulos": df["V1"].notnull().sum(),
                "Nulos": df["V1"].isnull().sum(),
                "% nulos": round(df["V1"].isnull().mean() * 100, 4),
                "Valores únicos": df["V1"].nunique(),
                "Memória MB": round(
                    df["V1"].memory_usage(
                        deep=True,
                        index=False
                    ) / (1024 ** 2),
                    4
                )
            }
        )

    if "tempo_desde_a_primeira_transacao" in df.columns:
        linhas_resumo_tcc.append(
            {
                "Variável": "tempo_desde_a_primeira_transacao",
                "Tipo": str(df["tempo_desde_a_primeira_transacao"].dtype),
                "Não nulos": df["tempo_desde_a_primeira_transacao"].notnull().sum(),
                "Nulos": df["tempo_desde_a_primeira_transacao"].isnull().sum(),
                "% nulos": round(df["tempo_desde_a_primeira_transacao"].isnull().mean() * 100, 4),
                "Valores únicos": df["tempo_desde_a_primeira_transacao"].nunique(),
                "Memória MB": round(
                    df["tempo_desde_a_primeira_transacao"].memory_usage(
                        deep=True,
                        index=False
                    ) / (1024 ** 2),
                    4
                )
            }
        )

    if "valor_de_transacao" in df.columns:
        linhas_resumo_tcc.append(
            {
                "Variável": "valor_de_transacao",
                "Tipo": str(df["valor_de_transacao"].dtype),
                "Não nulos": df["valor_de_transacao"].notnull().sum(),
                "Nulos": df["valor_de_transacao"].isnull().sum(),
                "% nulos": round(df["valor_de_transacao"].isnull().mean() * 100, 4),
                "Valores únicos": df["valor_de_transacao"].nunique(),
                "Memória MB": round(
                    df["valor_de_transacao"].memory_usage(
                        deep=True,
                        index=False
                    ) / (1024 ** 2),
                    4
                )
            }
        )

    if "status_fraude" in df.columns:
        linhas_resumo_tcc.append(
            {
                "Variável": "status_fraude",
                "Tipo": str(df["status_fraude"].dtype),
                "Não nulos": df["status_fraude"].notnull().sum(),
                "Nulos": df["status_fraude"].isnull().sum(),
                "% nulos": round(df["status_fraude"].isnull().mean() * 100, 4),
                "Valores únicos": df["status_fraude"].nunique(),
                "Memória MB": round(
                    df["status_fraude"].memory_usage(
                        deep=True,
                        index=False
                    ) / (1024 ** 2),
                    4
                )
            }
        )

    resumo_tcc_raw = pd.DataFrame(linhas_resumo_tcc)

    if not resumo_tcc_raw.empty:
        resumo_tcc_formatado = resumo_tcc_raw.copy()

        for col in ["Não nulos", "Nulos", "Valores únicos"]:
            resumo_tcc_formatado[col] = resumo_tcc_formatado[col].apply(formatar_int)

        resumo_tcc_formatado["% nulos"] = resumo_tcc_formatado["% nulos"].apply(
            lambda x: f"{x:.4f}%"
        )

        resumo_tcc_formatado["Memória MB"] = resumo_tcc_formatado["Memória MB"].apply(
            lambda x: f"{x:.4f}"
        )

    else:
        resumo_tcc_formatado = pd.DataFrame()

    # ========================================================
    # 10. PRIMEIRAS LINHAS
    # ========================================================

    primeiras_linhas = df.head(10).copy()

    # ========================================================
    # 11. EXPORTAÇÃO LaTeX
    # ========================================================

    if exportar_latex:
        exportar_artefatos_latex(
            pasta_latex=caminho_latex,
            resumo_etl=resumo_etl,
            tabela_transformacoes=tabela_transformacoes,
            resumo_status=resumo_status_raw,
            resumo_tcc=resumo_tcc_raw,
            estrutura_variaveis=estrutura_variaveis_raw,
            primeiras_linhas=primeiras_linhas
        )

    # ========================================================
    # 12. HTML DAS TABELAS
    # ========================================================

    tabela_resumo_etl_html = gerar_tabela_html(
        resumo_etl,
        table_id="tabela_resumo_etl"
    )

    tabela_transformacoes_html = gerar_tabela_html(
        tabela_transformacoes,
        table_id="tabela_transformacoes"
    )

    tabela_status_html = gerar_tabela_html(
        resumo_status,
        table_id="tabela_distribuicao_target"
    ) if not resumo_status.empty else "<p>Coluna status_fraude não encontrada.</p>"

    tabela_estrutura_html = gerar_tabela_html(
        estrutura_variaveis_formatada,
        table_id="tabela_estrutura_variaveis"
    )

    tabela_resumo_tcc_html = gerar_tabela_html(
        resumo_tcc_formatado,
        table_id="tabela_resumo_tcc"
    ) if not resumo_tcc_formatado.empty else "<p>Resumo agrupado não pôde ser gerado.</p>"

    tabela_primeiras_linhas_html = gerar_tabela_html(
        primeiras_linhas,
        table_id="tabela_primeiras_linhas"
    )

    colunas_originais_html = "".join(
        [
            f"<span class='tag'>{html.escape(str(col))}</span>"
            for col in colunas_originais
        ]
    )

    colunas_renomeadas_html = "".join(
        [
            f"<span class='tag'>{html.escape(str(col))}</span>"
            for col in colunas_apos_renomeacao
        ]
    )

    # ========================================================
    # 13. HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório de Estrutura da Base</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1350px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 10px;
                color: #020617;
                font-size: 34px;
            }}

            .subtitle {{
                text-align: center;
                color: #475569;
                font-size: 16px;
                margin-bottom: 34px;
            }}

            .section {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .section h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 24px;
                color: #020617;
            }}

            .section h3 {{
                margin-top: 26px;
                margin-bottom: 14px;
                color: #0f172a;
                font-size: 18px;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .metric-grid {{
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 14px;
                margin-top: 18px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .card-title {{
                font-size: 13px;
                font-weight: 800;
                color: #475569;
                margin-bottom: 8px;
            }}

            .card-value {{
                font-size: 23px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .card-desc {{
                font-size: 12px;
                color: #64748b;
                margin-top: 8px;
                line-height: 1.35;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 580px;
                overflow-y: auto;
            }}

            .tag-area {{
                display: flex;
                flex-wrap: wrap;
                gap: 8px;
                margin-top: 10px;
            }}

            .tag {{
                display: inline-block;
                background: #eff6ff;
                color: #1e3a8a;
                border: 1px solid #bfdbfe;
                border-radius: 999px;
                padding: 6px 10px;
                font-size: 12px;
                font-weight: 700;
            }}

            .pipeline {{
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 12px;
                margin-top: 18px;
            }}

            .step {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
                text-align: center;
            }}

            .step-number {{
                display: inline-flex;
                align-items: center;
                justify-content: center;
                width: 34px;
                height: 34px;
                border-radius: 999px;
                background: #2563eb;
                color: white;
                font-weight: 900;
                margin-bottom: 8px;
            }}

            .step-title {{
                font-size: 14px;
                font-weight: 900;
                color: #020617;
                margin-bottom: 6px;
            }}

            .step-desc {{
                font-size: 12px;
                color: #475569;
                line-height: 1.35;
            }}

            .note {{
                background: #fffbeb;
                border: 1px solid #fde68a;
                color: #78350f;
                border-radius: 12px;
                padding: 14px 16px;
                margin-top: 16px;
                font-size: 14px;
                line-height: 1.45;
            }}

            @media (max-width: 1000px) {{
                .metric-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .pipeline {{
                    grid-template-columns: repeat(2, 1fr);
                }}
            }}

            @media (max-width: 640px) {{
                .metric-grid {{
                    grid-template-columns: 1fr;
                }}

                .pipeline {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório de Estrutura da Base</h1>
            <div class="subtitle">
                ETL inicial, limpeza da base, renomeação das variáveis e análise estrutural dos campos.
            </div>

            <section class="section">
                <h2>Fluxo do Tratamento</h2>

                <div class="pipeline">
                    <div class="step">
                        <div class="step-number">1</div>
                        <div class="step-title">Leitura</div>
                        <div class="step-desc">Arquivo original lido: <strong>{arquivo_entrada}</strong></div>
                    </div>

                    <div class="step">
                        <div class="step-number">2</div>
                        <div class="step-title">Renomeação</div>
                        <div class="step-desc">Time, Amount e Class foram renomeadas para nomes descritivos.</div>
                    </div>

                    <div class="step">
                        <div class="step-number">3</div>
                        <div class="step-title">Limpeza</div>
                        <div class="step-desc">Remoção de duplicatas e linhas com valores faltantes.</div>
                    </div>

                    <div class="step">
                        <div class="step-number">4</div>
                        <div class="step-title">Exportação</div>
                        <div class="step-desc">Base tratada salva como <strong>{arquivo_saida_base}</strong>.</div>
                    </div>
                </div>
            </section>

            <section class="section">
                <h2>Resumo Geral do ETL</h2>

                <div class="metric-grid">
                    {gerar_card("Linhas iniciais", formatar_int(linhas_iniciais), "Quantidade de registros antes do tratamento.")}
                    {gerar_card("Colunas iniciais", formatar_int(colunas_iniciais), "Quantidade de variáveis no arquivo original.")}
                    {gerar_card("Linhas finais", formatar_int(linhas_finais), "Quantidade de registros após limpeza.")}
                    {gerar_card("Colunas finais", formatar_int(colunas_finais), "Quantidade de variáveis após renomeação.")}
                    {gerar_card("Duplicatas identificadas", formatar_int(duplicatas_identificadas), "Número de linhas duplicadas detectadas.")}
                    {gerar_card("Removidas por duplicidade", formatar_int(linhas_removidas_duplicatas), f"{percentual_duplicatas_removidas:.4f}% das linhas iniciais.")}
                    {gerar_card("Linhas com faltantes", formatar_int(linhas_com_faltantes), "Linhas com pelo menos um valor ausente.")}
                    {gerar_card("Removidas por faltantes", formatar_int(linhas_removidas_faltantes), f"{percentual_faltantes_removidas:.4f}% das linhas iniciais.")}
                    {gerar_card("Total removido", formatar_int(linhas_removidas_total), f"{percentual_linhas_removidas:.4f}% das linhas iniciais.")}
                    {gerar_card("Base preservada", f"{percentual_base_preservada:.4f}%", "Percentual da base original mantida após limpeza.")}
                    {gerar_card("Arquivo tratado", arquivo_saida_base, "Nome da base final exportada.")}
                    {gerar_card("Dimensão final", f"{formatar_int(linhas_finais)} x {formatar_int(colunas_finais)}", "Linhas x colunas.")}
                </div>
            </section>

            {gerar_secao_tabela(
                "Tabela Resumo do ETL",
                tabela_resumo_etl_html,
                "tabela_resumo_etl",
                "tabela_resumo_etl.csv"
            )}

            {gerar_secao_tabela(
                "Tabela de Transformações Aplicadas",
                tabela_transformacoes_html,
                "tabela_transformacoes",
                "tabela_transformacoes.csv"
            )}

            <section class="section">
                <h2>Renomeação das Colunas</h2>

                <h3>Colunas originais</h3>
                <div class="tag-area">
                    {colunas_originais_html}
                </div>

                <h3>Colunas após renomeação</h3>
                <div class="tag-area">
                    {colunas_renomeadas_html}
                </div>

                <div class="note">
                    As colunas <strong>Time</strong>, <strong>Amount</strong> e <strong>Class</strong>
                    foram renomeadas para <strong>tempo_desde_a_primeira_transacao</strong>,
                    <strong>valor_de_transacao</strong> e <strong>status_fraude</strong>, respectivamente.
                </div>
            </section>

            {gerar_secao_tabela(
                "Distribuição do Target após Tratamento",
                tabela_status_html,
                "tabela_distribuicao_target",
                "tabela_distribuicao_target.csv"
            )}

            {gerar_secao_tabela(
                "Resumo Agrupado para o TCC",
                tabela_resumo_tcc_html,
                "tabela_resumo_tcc",
                "tabela_resumo_tcc.csv"
            )}

            {gerar_secao_tabela(
                "Estrutura das Variáveis",
                tabela_estrutura_html,
                "tabela_estrutura_variaveis",
                "tabela_estrutura_variaveis.csv"
            )}

            {gerar_secao_tabela(
                "Primeiras Linhas da Base Tratada",
                tabela_primeiras_linhas_html,
                "tabela_primeiras_linhas",
                "tabela_primeiras_linhas.csv"
            )}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    # ========================================================
    # 14. SALVAR HTML
    # ========================================================

    caminho_html.write_text(
        html_final,
        encoding="utf-8"
    )

    print("=" * 80)
    print("RELATÓRIO GERADO COM SUCESSO")
    print("=" * 80)
    print(f"Base tratada salva em: {caminho_base_tratada.resolve()}")
    print(f"Relatório HTML salvo em: {caminho_html.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX salvos em: {caminho_latex.resolve()}")

    print("=" * 80)

    return {
        "caminho_html": caminho_html,
        "caminho_base": caminho_base_tratada,
        "caminho_latex": caminho_latex,
        "resumo_etl": resumo_etl,
        "tabela_transformacoes": tabela_transformacoes,
        "resumo_status": resumo_status_raw,
        "estrutura_variaveis": estrutura_variaveis_raw,
        "resumo_tcc": resumo_tcc_raw,
        "primeiras_linhas": primeiras_linhas
    }


# ============================================================
# CHAMADA
# ============================================================

resultado_relatorio_estrutura = gerar_relatorio_estrutura_base_html(
    arquivo_entrada="a.csv",
    arquivo_saida_base="creditcard.csv",
    arquivo_html_saida="relatorio_estrutura_base.html",
    pasta_saida=".",
    pasta_latex="relatorio_estrutura_base",
    exportar_latex=True
)

resultado_relatorio_estrutura["caminho_html"]


RELATÓRIO GERADO COM SUCESSO
Base tratada salva em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\creditcard.csv
Relatório HTML salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\relatorio_estrutura_base.html
Arquivos LaTeX salvos em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\relatorio_estrutura_base


WindowsPath('relatorio_estrutura_base.html')